# Tables

## Forecast daily calendar metric

In [0]:
%sql
SELECT COUNT(*) FROM samples.accuweather.forecast_daily_calendar_metric

In [0]:
%sql
SELECT * FROM samples.accuweather.forecast_daily_calendar_metric

### point_table

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW point_table AS
SELECT * FROM VALUES 
    (34.56, 137.2), --tokyo, jp
    (13.6, 100.23), -- bangkok, th
    (28.63, 76.3), -- delhi, in
    (28.64336,77.11756)
AS t(latitude, longitude);

In [0]:
%sql
SELECT * FROM point_table

In [0]:
%sql
WITH DistanceCalculations AS (
  SELECT 
    p.longitude,
    p.latitude,
    s.city_name AS site_column,
    s.country_code AS country,
    -- Calculates spherical distance in meters between the two points
    ST_DistanceSphere(
      ST_Point(p.longitude, p.latitude),
      ST_Point(s.longitude, s.latitude)
    ) AS distance_meters,
    ROW_NUMBER() OVER (
      PARTITION BY p.longitude, p.latitude 
      ORDER BY ST_DistanceSphere(
        ST_Point(p.longitude, p.latitude),
        ST_Point(s.longitude, s.latitude)
      ) ASC
    ) AS rank
  FROM point_table p
  CROSS JOIN samples.accuweather.forecast_daily_calendar_metric s
)
SELECT 
  longitude,
  latitude,
  site_column,
  country,
  distance_meters
FROM DistanceCalculations
WHERE rank = 1;

In [0]:
%sql
SELECT 
  p.longitude,
  p.latitude,
  s.city_name AS site_column,
  s.country_code AS country,
  -- Calculates spherical distance in meters between the two points
  ST_DistanceSphere(
    ST_Point(p.longitude, p.latitude),
    ST_Point(s.longitude, s.latitude)
  ) AS distance_meters
FROM point_table p
LEFT JOIN samples.accuweather.forecast_daily_calendar_metric s
  -- Converts (lat, lon) to a hexagonal H3 cell ID at resolution 8 (~0.7 km²)
  ON h3_longlatash3(p.longitude, p.latitude, 8) = h3_longlatash3(s.longitude, s.latitude, 8);

In [0]:
%sql
SELECT DISTINCT
  p.longitude,
  p.latitude,
  s.city_name AS site_column,
  ST_DistanceSphere(
    ST_Point(p.longitude, p.latitude),
    ST_Point(s.longitude, s.latitude)
  ) AS distance_meters
FROM point_table p
CROSS JOIN (
  SELECT city_name, longitude, latitude
  FROM samples.accuweather.forecast_daily_calendar_metric
) s;

# References
* Geospatial Insights With Databricks SQL: Techniques and Applications: https://www.youtube.com/watch?v=0ENStcXva1A